# 盈利因子 (RMW) 完整教程：从 ROE 分组到多空组合检验

## 📚 教学目标
1. 理解**盈利因子**的定义：按 ROE 等盈利指标分组，高盈利组 − 低盈利组 = RMW
2. 掌握 **Fama-French 五因子**中 RMW（Robust Minus Weak）的构建方法
3. 在**微型数据集**（10 只股票 × 1 月）上手算 ROE 分组和 Spread
4. 扩展到 **300 只股票 × 60 月**，完成 T 检验和单调性检验
5. 理解 **A 股盈利因子**的特殊性：等权 vs 市值加权的差异、与市值因子的交互

**参考书**：《因子投资：方法与实践》（石川）第 3.6 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 什么是盈利因子？为什么高盈利股票收益更高？

### 🎯 一个直觉问题

假设你面前有两家公司：A 公司 ROE=20%，B 公司 ROE=3%。其他条件相同，你更愿意买哪家公司的股票？

直觉告诉我们应该买盈利能力更强的 A 公司。**盈利因子 (RMW)** 就是检验这个直觉：做多高盈利股票、做空低盈利股票，能否获得显著的超额收益？

### 📖 书中的定义

> 盈利能力指企业利用已有资源获取利润的能力。通常来说，盈利能力越强，企业前景越好，也越容易受到投资者的关注。

> Fama and French（2015）将年度营业利润作为分子计算 ROE，而 Hou et al.（2015）将季度净利润作为分子。这些研究均表明 ROE 代表的盈利能力与股票的未来收益呈显著正相关。

### 📐 盈利因子的理论基础

盈利因子的理论基础比其他因子更扎实：

| 理论来源 | 核心逻辑 |
|---------|----------|
| **DDM（股利贴现模型）** | 股价 = 未来盈利的贴现值 → 高盈利 → 高估值 → 高收益 |
| **q-理论（Hou et al.）** | 在均衡时，盈利率 / 投资边际成本 = 折现率 = 预期收益率 |
| **剩余收益模型 (RIM)** | 股价 = 账面价值 + 未来超额盈利的贴现值 |

### 📐 常用盈利指标

| 指标 | 公式 | 提出者 | 特点 |
|------|------|--------|------|
| **ROE** | 净利润 / 净资产 | 最常用 | FF5 采用 |
| **ROA** | 净利润 / 总资产 | 常用 | 剔除杠杆影响 |
| **GP** | 毛利润 / 总资产 | Novy-Marx (2013) | 预测能力与 BM 相当 |
| **ROTC** | EBIT / 有形资本 | Greenblatt | "神奇公式"的盈利指标 |
| **ROIC** | NOPAT / 投入资本 | Damodaran | 综合衡量 |
| **RNOA** | 经营利润 / 净经营资产 | Nissim & Penman | 剔除杠杆和非经营项目 |

### 📐 RMW 因子的定义

$$\text{RMW} = R_{\text{High Profit}} - R_{\text{Low Profit}}$$

其中：
- $R_{\text{High Profit}}$ = 高盈利组（如 ROE 最高的 30%）的组合收益率
- $R_{\text{Low Profit}}$ = 低盈利组（如 ROE 最低的 30%）的组合收益率
- RMW > 0 表示高盈利股票收益高于低盈利股票

In [ ]:
import sys, os
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")
try:
    import matplotlib
    print(f"matplotlib: {matplotlib.__version__}")
except ImportError:
    print("❌ matplotlib 未安装! 请运行: !pip install matplotlib seaborn statsmodels scipy")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 微型数据集：10 只股票的手算演示

### 🎯 场景

假设市场上只有 **10 只股票**，我们用 ROE（净资产收益率）作为盈利指标，检验高盈利股票是否获得更高收益。

### 📐 数据生成逻辑

$$r_i = \bar{r} + \gamma \cdot (\text{ROE}_i - \overline{\text{ROE}}) + \varepsilon_i$$

其中：
- $\bar{r}$ = 基础收益率（所有股票共享）
- $\gamma$ = 盈利效应系数（> 0 表示高盈利带来高收益）
- $\varepsilon_i$ = 个股噪声

In [ ]:
# ========== 构造 10 只股票的微型数据 ==========
np.random.seed(42)

stocks = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10']

# ROE（净资产收益率，%）：从低到高
roe = np.array([-5, -1, 3, 6, 8, 10, 13, 16, 20, 28])

# 数据生成参数
base_return = 1.0    # 月基础收益率 1%
gamma = 0.12          # 盈利效应：每 1% ROE → 0.12% 额外收益
noise = np.random.normal(0, 2.0, 10)

# 收益率 = 基础 + 盈利效应 + 噪声
returns = base_return + gamma * (roe - roe.mean()) + noise

# 构建 DataFrame
df_micro = pd.DataFrame({
    'Stock': stocks,
    'ROE (%)': roe,
    'Return (%)': np.round(returns, 2)
})

print("📋 10 只股票数据集：")
print(df_micro.to_string(index=False))
print(f"\n💡 盈利效应系数 γ = {gamma}（每 1% ROE → 本期多赚 {gamma}%）")
print(f"   基础收益率 = {base_return}%，噪声标准差 = 2.0%")

### 📐 步骤 1: 按 ROE 排序分组

将 10 只股票按 ROE 从低到高排序，分为 2 组：
- **Low 组（Q1）**：ROE 最低的 5 只 → 做空
- **High 组（Q2）**：ROE 最高的 5 只 → 做多

$$\text{RMW} = \bar{R}_{\text{High}} - \bar{R}_{\text{Low}}$$

In [ ]:
# ========== 步骤 1: 按 ROE 排序分组 ==========
print("📊 步骤 1: 按 ROE 排序分组")
print("─" * 55)

df_sorted = df_micro.sort_values('ROE (%)').reset_index(drop=True)
df_sorted['Group'] = ['Low'] * 5 + ['High'] * 5

print("\n  Low 盈利组（做空）:")
for _, row in df_sorted[df_sorted['Group'] == 'Low'].iterrows():
    print(f"    {row['Stock']}: ROE = {row['ROE (%)']:>5}%,  收益 = {row['Return (%)']:>6.2f}%")

print("\n  High 盈利组（做多）:")
for _, row in df_sorted[df_sorted['Group'] == 'High'].iterrows():
    print(f"    {row['Stock']}: ROE = {row['ROE (%)']:>5}%,  收益 = {row['Return (%)']:>6.2f}%")

In [ ]:
# ========== 步骤 2: 计算多空收益 (RMW Spread) ==========
print("\n📊 步骤 2: 构建 RMW 多空组合")
print("─" * 55)

ret_high = df_sorted[df_sorted['Group'] == 'High']['Return (%)'].mean()
ret_low = df_sorted[df_sorted['Group'] == 'Low']['Return (%)'].mean()
spread = ret_high - ret_low

print(f"  R_High (高盈利组) = {ret_high:.2f}%")
print(f"  R_Low  (低盈利组) = {ret_low:.2f}%")
print(f"  RMW Spread = R_High - R_Low = {ret_high:.2f} - {ret_low:.2f} = {spread:.2f}%")
print(f"\n💡 解读: RMW = {spread:.2f}% {'> 0 → 高盈利股票收益更高 ✅' if spread > 0 else '< 0 → 高盈利股票收益更低 ❌'}")

In [ ]:
# ========== 可视化: ROE 分组与收益 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: ROE vs 收益散点图
ax1 = axes[0]
colors_scatter = ['#e74c3c'] * 5 + ['#2ecc71'] * 5
ax1.scatter(df_micro['ROE (%)'], df_micro['Return (%)'], c=colors_scatter, s=100, edgecolors='black', zorder=5)
for _, row in df_micro.iterrows():
    ax1.annotate(row['Stock'], (row['ROE (%)'], row['Return (%)']), fontsize=9, ha='center', va='bottom')
# 拟合线
z = np.polyfit(df_micro['ROE (%)'], df_micro['Return (%)'], 1)
p = np.poly1d(z)
x_line = np.linspace(df_micro['ROE (%)'].min() - 2, df_micro['ROE (%)'].max() + 2, 100)
ax1.plot(x_line, p(x_line), '--', color='gray', alpha=0.7, label=f'Slope = {z[0]:.3f}')
ax1.set_xlabel('ROE (%)', fontsize=12)
ax1.set_ylabel('Return (%)', fontsize=12)
ax1.set_title('ROE vs Stock Return (Micro Example)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# 右图: 分组平均收益柱状图
ax2 = axes[1]
group_means = [ret_low, ret_high]
group_labels = ['Low ROE\n(Q1)', 'High ROE\n(Q2)']
bar_colors = ['#e74c3c', '#2ecc71']
bars = ax2.bar(group_labels, group_means, color=bar_colors, edgecolor='black', alpha=0.85, width=0.5)
for bar, v in zip(bars, group_means):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
             f'{v:.2f}%', ha='center', va='bottom', fontweight='bold')
# Spread 箭头
ax2.annotate('', xy=(1, ret_high + 0.02), xytext=(0, ret_low + 0.02),
             arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax2.text(0.5, (ret_high + ret_low) / 2, f'RMW\n{spread:.2f}%', ha='center', va='center',
         fontsize=12, fontweight='bold', color='blue',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='blue'))
ax2.set_ylabel('Average Return (%)', fontsize=12)
ax2.set_title('Profitability Factor (RMW) Spread', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：ROE 越高的股票，本期收益倾向越高（正斜率 = {z[0]:.3f}）")
print(f"  右图：High ROE 组收益 = {ret_high:.2f}%，Low ROE 组收益 = {ret_low:.2f}%")
print(f"  RMW Spread = {spread:.2f}%，即高盈利股票比低盈利股票多赚 {spread:.2f}%")

---

## 3. 五组分组：更细致的盈利分层

### 📐 将 10 只股票按 ROE 分成 5 组

在实际研究中，通常将股票按盈利指标分成 5 组或 10 组，以观察收益率是否呈现**单调递增**的特征。

$$\text{Spearman Rank Correlation} = 1 - \frac{6 \sum d_i^2}{n(n^2 - 1)}$$

其中 $d_i$ = 第 i 组的排名与收益率排名之差。Spearman 相关系数越接近 1，说明单调性越好。

In [ ]:
# ========== 步骤 3: 五组分组 ==========
print("📊 步骤 3: 将 10 只股票按 ROE 分成 5 组")
print("─" * 55)

df_sorted5 = df_micro.sort_values('ROE (%)').reset_index(drop=True)
group_labels_5 = ['Q1 (Low)', 'Q2', 'Q3', 'Q4', 'Q5 (High)']
df_sorted5['Quintile'] = np.repeat(group_labels_5, 2)

group_rets = df_sorted5.groupby('Quintile')['Return (%)'].mean()
group_roe = df_sorted5.groupby('Quintile')['ROE (%)'].mean()

print(f"\n  {'Quintile':<12} {'Avg ROE (%)':>12} {'Avg Return (%)':>15}")
print("  " + "─" * 42)
for q in group_labels_5:
    print(f"  {q:<12} {group_roe[q]:>12.1f} {group_rets[q]:>15.2f}")

# Spearman 单调性检验
group_ranks = np.arange(1, 6)
ret_values = np.array([group_rets[q] for q in group_labels_5])
spearman_corr, spearman_p = stats.spearmanr(group_ranks, ret_values)

print(f"\n📊 单调性检验:")
print(f"  Spearman Rank Correlation = {spearman_corr:.4f}")
print(f"  p-value = {spearman_p:.4f}")
print(f"  {'✅ 强单调性' if abs(spearman_corr) > 0.8 else '⚠️ 单调性一般' if abs(spearman_corr) > 0.5 else '❌ 单调性弱'}")

In [ ]:
# ========== 可视化: 五组收益 ==========
fig, ax = plt.subplots(figsize=(10, 6))

colors_5 = plt.cm.RdYlGn(np.linspace(0.15, 0.85, 5))
bars = ax.bar(group_labels_5, ret_values, color=colors_5, edgecolor='black', alpha=0.85, width=0.6)

for bar, v in zip(bars, ret_values):
    va = 'bottom' if v >= 0 else 'top'
    offset = 0.05 if v >= 0 else -0.05
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
            f'{v:.2f}%', ha='center', va=va, fontweight='bold')

ax.set_xlabel('ROE Quintile', fontsize=12)
ax.set_ylabel('Average Return (%)', fontsize=12)
ax.set_title('Return by ROE Quintile (Micro Example)', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# 标注 Spearman
textstr = f'Spearman \u03c1 = {spearman_corr:.3f}\np-value = {spearman_p:.3f}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.98, 0.98, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', horizontalalignment='right', bbox=props)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  五组收益从 Q1 到 Q5 呈 {'单调递增' if spearman_corr > 0.5 else '非单调'} 趋势")
print(f"  Spearman 相关系数 = {spearman_corr:.3f}，说明 ROE 对收益有 {'强' if abs(spearman_corr) > 0.8 else '中等'} 预测能力")

---

## 4. 扩展到完整规模：300 只股票 × 60 个月

### 📐 模拟参数

将微型示例扩展到更接近真实市场的规模：
- 300 只股票，每月重新按 ROE 分组
- 60 个月的时间跨度
- 每月计算 RMW Spread，最后做 T 检验

### 📐 数据生成过程 (DGP)

$$r_{i,t} = \bar{r}_t + \gamma \cdot (\text{ROE}_{i,t} - \overline{\text{ROE}}_t) + \sigma_{\varepsilon} \cdot \varepsilon_{i,t}$$

其中 $\gamma > 0$ 表示盈利效应存在。

In [ ]:
# ========== 模拟完整数据集 ==========
np.random.seed(42)

N_STOCKS = 300
N_MONTHS = 60
N_GROUPS = 5

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"  分组数量: {N_GROUPS} 组")
print(f"\n  DGP: r_{{i,t}} = base + 0.08 × (ROE_{{i,t}} - mean) + noise")

gamma_full = 0.08  # 盈利效应系数

# 存储每月的分组收益率
monthly_group_rets = {f'Q{i}': [] for i in range(1, N_GROUPS + 1)}
monthly_spreads = []

for t in range(N_MONTHS):
    # 每月生成截面数据
    base_ret = np.random.normal(1.0, 3.0)  # 月基础收益
    roe_t = np.random.normal(10, 8, N_STOCKS)  # ROE ~ N(10%, 8%)
    noise_t = np.random.normal(0, 5, N_STOCKS)
    ret_t = base_ret + gamma_full * (roe_t - roe_t.mean()) + noise_t

    # 按 ROE 分组
    df_t = pd.DataFrame({'ROE': roe_t, 'Return': ret_t})
    df_t['Quintile'] = pd.qcut(df_t['ROE'], N_GROUPS, labels=[f'Q{i}' for i in range(1, N_GROUPS + 1)])

    # 计算各组平均收益
    group_ret = df_t.groupby('Quintile', observed=True)['Return'].mean()
    for q in range(1, N_GROUPS + 1):
        monthly_group_rets[f'Q{q}'].append(group_ret[f'Q{q}'])

    # RMW Spread = Q5 (High ROE) - Q1 (Low ROE)
    spread_t = group_ret['Q5'] - group_ret['Q1']
    monthly_spreads.append(spread_t)

print(f"\n✅ 数据模拟完成：{N_STOCKS} 只股票 × {N_MONTHS} 个月")
print(f"  每月计算 {N_GROUPS} 组收益 + RMW Spread")

### 📐 步骤 4: T 检验——盈利因子是否显著？

对 60 个月的 RMW Spread 时间序列做单样本 T 检验：

$$t = \frac{\overline{\text{Spread}}}{s_{\text{Spread}} / \sqrt{T}}$$

其中：
- $\overline{\text{Spread}}$ = 60 个月 RMW Spread 的均值
- $s_{\text{Spread}}$ = 样本标准差
- $T = 60$ = 月数
- 如果 $|t| > 2.0$（近似），则在 5% 水平下显著

In [ ]:
# ========== 步骤 4: T 检验 ==========
print("📊 步骤 4: RMW Spread 的 T 检验")
print("─" * 55)

spreads = np.array(monthly_spreads)
spread_mean = spreads.mean()
spread_std = spreads.std(ddof=1)
spread_se = spread_std / np.sqrt(N_MONTHS)
t_stat = spread_mean / spread_se
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=N_MONTHS - 1))

print(f"\n  📐 手算 T 检验:")
print(f"  Spread 均值  = {spread_mean:.4f}%")
print(f"  Spread 标准差 = {spread_std:.4f}%")
print(f"  标准误 (SE)  = {spread_std:.4f} / √{N_MONTHS} = {spread_se:.4f}")
print(f"  T 统计量    = {spread_mean:.4f} / {spread_se:.4f} = {t_stat:.4f}")
print(f"  P 值 (双尾)  = {p_value:.6f}")

# scipy 验证
t_scipy, p_scipy = stats.ttest_1samp(spreads, 0)
print(f"\n  🔬 scipy 验证:")
print(f"  scipy T = {t_scipy:.6f}")
print(f"  scipy P = {p_scipy:.6f}")
print(f"  ✅ 验证通过！" if abs(t_stat - t_scipy) < 0.001 else "  ❌ 验证失败！")

# 显著性判断
alpha_level = 0.05
print(f"\n  🎯 结论 (α = {alpha_level}):")
if p_value < alpha_level:
    print(f"  ✅ RMW 显著！t = {t_stat:.2f}, p = {p_value:.4f} < {alpha_level}")
    print(f"  高盈利股票收益显著高于低盈利股票")
else:
    print(f"  ❌ RMW 不显著。t = {t_stat:.2f}, p = {p_value:.4f} >= {alpha_level}")
    print(f"  高盈利股票收益与低盈利股票无显著差异")

In [ ]:
# ========== 步骤 5: 各组平均收益和单调性 ==========
print("📊 步骤 5: 各组平均收益率和单调性检验")
print("─" * 55)

avg_rets = {q: np.mean(monthly_group_rets[q]) for q in [f'Q{i}' for i in range(1, N_GROUPS + 1)]}
ret_values_full = np.array([avg_rets[f'Q{i}'] for i in range(1, N_GROUPS + 1)])

print(f"\n  {'Quintile':<10} {'Avg Monthly Return (%)':>22}")
print("  " + "─" * 35)
for i in range(1, N_GROUPS + 1):
    print(f"  Q{i:<9} {avg_rets[f'Q{i}']:>22.4f}")

# Spearman 检验
spearman_full, spearman_p_full = stats.spearmanr(np.arange(1, N_GROUPS + 1), ret_values_full)
print(f"\n  Spearman Rank Correlation = {spearman_full:.4f}")
print(f"  p-value = {spearman_p_full:.4f}")
if abs(spearman_full) > 0.8:
    print(f"  ✅ 强单调性：ROE 越高，收益越高")
elif abs(spearman_full) > 0.5:
    print(f"  ⚠️ 中等单调性")
else:
    print(f"  ❌ 单调性弱")

In [ ]:
# ========== 经济显著性：Sharpe Ratio ==========
print("📊 经济显著性: RMW 的 Sharpe Ratio")
print("─" * 55)

sr_monthly = spread_mean / spread_std
sr_annual = sr_monthly * np.sqrt(12)

print(f"  月均 Spread   = {spread_mean:.4f}%")
print(f"  月 Sharpe Ratio = {spread_mean:.4f} / {spread_std:.4f} = {sr_monthly:.4f}")
print(f"  年化 Sharpe Ratio = {sr_monthly:.4f} × √12 = {sr_annual:.4f}")
print(f"\n  📐 验证: t = SR_monthly × √T = {sr_monthly:.4f} × √{N_MONTHS} = {sr_monthly * np.sqrt(N_MONTHS):.4f}")
print(f"  （应等于 T 统计量 {t_stat:.4f}）")

# 置信区间
t_critical = stats.t.ppf(0.975, df=N_MONTHS - 1)
margin = t_critical * spread_se
ci_lower = spread_mean - margin
ci_upper = spread_mean + margin
print(f"\n  95% 置信区间: [{ci_lower:.4f}%, {ci_upper:.4f}%]")

In [ ]:
# ========== 可视化: 完整规模结果 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 左图: 各组平均收益
ax1 = axes[0]
q_labels = [f'Q{i}' for i in range(1, N_GROUPS + 1)]
colors_q = plt.cm.RdYlGn(np.linspace(0.15, 0.85, N_GROUPS))
bars1 = ax1.bar(q_labels, ret_values_full, color=colors_q, edgecolor='black', alpha=0.85)
for bar, v in zip(bars1, ret_values_full):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.set_xlabel('ROE Quintile', fontsize=12)
ax1.set_ylabel('Avg Monthly Return (%)', fontsize=12)
ax1.set_title('Return by ROE Quintile (300 Stocks x 60 Months)', fontsize=13, fontweight='bold')
textstr1 = f'Spearman \u03c1 = {spearman_full:.3f}'
props1 = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax1.text(0.98, 0.98, textstr1, transform=ax1.transAxes, fontsize=10,
         verticalalignment='top', horizontalalignment='right', bbox=props1)
ax1.grid(axis='y', alpha=0.3)

# 中图: RMW Spread 时间序列
ax2 = axes[1]
ax2.plot(range(N_MONTHS), spreads, color='steelblue', alpha=0.7, linewidth=1)
ax2.axhline(y=spread_mean, color='red', linestyle='--', linewidth=2, label=f'Mean = {spread_mean:.3f}%')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.fill_between(range(N_MONTHS), ci_lower, ci_upper, alpha=0.15, color='red', label='95% CI')
ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('RMW Spread (%)', fontsize=12)
ax2.set_title('RMW Spread Time Series', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

# 右图: T 检验结果
ax3 = axes[2]
x_t = np.linspace(-4, 8, 500)
y_t = stats.t.pdf(x_t, df=N_MONTHS - 1)
ax3.plot(x_t, y_t, 'b-', linewidth=2, label=f't-distribution (df={N_MONTHS-1})')
ax3.fill_between(x_t, y_t, where=(x_t > 2.0) | (x_t < -2.0), alpha=0.3, color='red', label='Rejection region (5%)')
ax3.axvline(x=t_stat, color='green', linewidth=2, linestyle='--', label=f't = {t_stat:.2f}')
ax3.set_xlabel('t Value', fontsize=12)
ax3.set_ylabel('Probability Density', fontsize=12)
ax3.set_title('T-Test for RMW Spread', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：ROE 五组收益 {'呈现单调递增' if spearman_full > 0.5 else '未呈现明显单调性'}")
print(f"  中图：RMW Spread 时间序列，均值 = {spread_mean:.3f}%，95% CI {'不包含 0 → 显著' if ci_lower > 0 or ci_upper < 0 else '包含 0 → 不显著'}")
print(f"  右图：T 统计量 = {t_stat:.2f}，{'落在拒绝域 → 显著' if abs(t_stat) > 2.0 else '未落在拒绝域 → 不显著'}")

---

## 5. 盈利因子与市值因子的交互：双重排序

### 📖 书中的关键发现

> 等权重和市值加权的结果差异说明，等权重配置提升了低 ROE（TTM）组的月均收益率，表明低 ROE（TTM）的公司在小市值上有正的暴露，即低 ROE 的公司更可能是小市值的公司。

> 控制公司的质量可以使规模因子（小市值效应）在美股上更加显著。小市值中有很多盈利能力非常差的股票，这些股票的低收益削弱了小市值效应。

### 📐 双重排序方法

1. 先按市值分成 3 组（Small / Medium / Big）
2. 在每个市值组内，再按 ROE 分成 5 组
3. 在每个市值组内计算 RMW Spread
4. 最后取平均

这样可以控制市值对盈利因子的干扰。

In [ ]:
# ========== 双重排序: 市值 × ROE ==========
np.random.seed(42)

print("📊 双重排序: 控制市值后检验盈利因子")
print("─" * 55)

# 模拟数据：加入市值变量
monthly_double_sort = {'Small': [], 'Medium': [], 'Big': []}

for t in range(N_MONTHS):
    base_ret = np.random.normal(1.0, 3.0)
    roe_t = np.random.normal(10, 8, N_STOCKS)
    # 市值与 ROE 正相关（高 ROE 公司往往市值更大）
    mktcap_t = np.exp(np.random.normal(10, 1.5, N_STOCKS) + 0.03 * roe_t)
    noise_t = np.random.normal(0, 5, N_STOCKS)
    ret_t = base_ret + gamma_full * (roe_t - roe_t.mean()) + noise_t

    df_t = pd.DataFrame({'ROE': roe_t, 'MktCap': mktcap_t, 'Return': ret_t})

    # 第一步：按市值分 3 组
    df_t['SizeGroup'] = pd.qcut(df_t['MktCap'], 3, labels=['Small', 'Medium', 'Big'])

    # 第二步：在每个市值组内按 ROE 分 5 组
    def roe_sort(g):
        g = g.copy()
        g['ROEQuintile'] = pd.qcut(g['ROE'], 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
        return g

    df_t = df_t.groupby('SizeGroup', group_keys=False, observed=True).apply(roe_sort)

    # 在每个市值组内计算 RMW
    for sg in ['Small', 'Medium', 'Big']:
        sub = df_t[df_t['SizeGroup'] == sg]
        ret_q5 = sub[sub['ROEQuintile'] == 'Q5']['Return'].mean()
        ret_q1 = sub[sub['ROEQuintile'] == 'Q1']['Return'].mean()
        monthly_double_sort[sg].append(ret_q5 - ret_q1)

# 计算各市值组的 RMW 统计量
print(f"\n  {'Size Group':<12} {'RMW Mean (%)':>14} {'t-stat':>10} {'Significant':>12}")
print("  " + "─" * 52)

all_double_spreads = []
for sg in ['Small', 'Medium', 'Big']:
    arr = np.array(monthly_double_sort[sg])
    m = arr.mean()
    se = arr.std(ddof=1) / np.sqrt(N_MONTHS)
    t = m / se
    p = 2 * (1 - stats.t.cdf(abs(t), df=N_MONTHS - 1))
    sig = "✅" if p < 0.05 else "❌"
    print(f"  {sg:<12} {m:>14.4f} {t:>10.2f} {sig:>12}")
    all_double_spreads.extend(arr.tolist())

# 平均 RMW
avg_rmw = np.mean(all_double_spreads)
print(f"\n  📊 控制市值后平均 RMW = {avg_rmw:.4f}%")

In [ ]:
# ========== 可视化: 双重排序结果 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: 各市值组的 RMW
ax1 = axes[0]
sg_labels = ['Small', 'Medium', 'Big']
sg_means = [np.array(monthly_double_sort[sg]).mean() for sg in sg_labels]
sg_tstats = [np.array(monthly_double_sort[sg]).mean() / (np.array(monthly_double_sort[sg]).std(ddof=1) / np.sqrt(N_MONTHS)) for sg in sg_labels]

bar_colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax1.bar(sg_labels, sg_means, color=bar_colors, edgecolor='black', alpha=0.85, width=0.5)
for bar, v, t in zip(bars, sg_means, sg_tstats):
    va = 'bottom' if v >= 0 else 'top'
    offset = 0.02 if v >= 0 else -0.02
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
             f'{v:.3f}%\nt={t:.2f}', ha='center', va=va, fontweight='bold', fontsize=9)
ax1.axhline(y=0, color='black', linewidth=0.5)
ax1.set_xlabel('Size Group', fontsize=12)
ax1.set_ylabel('RMW Spread (%)', fontsize=12)
ax1.set_title('Profitability Factor by Size Group', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# 右图: RMW 累计收益
ax2 = axes[1]
for sg, color in zip(sg_labels, bar_colors):
    cumret = np.cumsum(monthly_double_sort[sg])
    ax2.plot(range(N_MONTHS), cumret, label=sg, color=color, linewidth=1.5)
avg_cumret = np.cumsum(np.mean([monthly_double_sort[sg] for sg in sg_labels], axis=0))
ax2.plot(range(N_MONTHS), avg_cumret, label='Average', color='black', linewidth=2, linestyle='--')
ax2.axhline(y=0, color='gray', linewidth=0.5)
ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('Cumulative Return (%)', fontsize=12)
ax2.set_title('Cumulative RMW Return by Size Group', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：控制市值后，各市值组内 RMW 均 {'为正' if all(m > 0 for m in sg_means) else '有正有负'}")
print(f"  右图：各市值组 RMW 累计收益曲线")
print(f"  关键发现：等权 vs 市值加权的差异源于低 ROE 公司往往是小市值公司")

---

## 6. 等权 vs 市值加权的差异

### 📖 书中的发现

> 当采用等权重构造投资组合时，这 10 个投资组合的月均收益率虽然表现出较好的单调性，ROE（TTM）和收益率的秩相关系数为 0.855，但差异很小，导致 High-Low 组合的盈利因子月均收益率非常不显著。而当采用市值加权时，单调性有所下降（秩相关系数为 0.552），但差异非常突出，盈利因子的月均收益率的显著性有所上升。

### 💡 为什么会这样？

| 加权方式 | 特点 | 对盈利因子的影响 |
|---------|------|------------------|
| **等权重** | 每只股票权重相同 | 低 ROE 组中的小市值股票被提升权重 → 低 ROE 组收益变高 → Spread 缩小 |
| **市值加权** | 大市值权重更大 | 高 ROE 组中的大市值股票权重更大 → 高 ROE 组收益更高 → Spread 扩大 |

本质原因：**ROE 与市值正相关**，低 ROE 公司往往是小市值公司。

In [ ]:
# ========== 等权 vs 市值加权对比 ==========
np.random.seed(42)

print("📊 等权 vs 市值加权的盈利因子表现")
print("─" * 55)

ew_spreads = []
vw_spreads = []

for t in range(N_MONTHS):
    base_ret = np.random.normal(1.0, 3.0)
    roe_t = np.random.normal(10, 8, N_STOCKS)
    # 市值与 ROE 正相关
    mktcap_t = np.exp(np.random.normal(10, 1.5, N_STOCKS) + 0.03 * roe_t)
    noise_t = np.random.normal(0, 5, N_STOCKS)
    ret_t = base_ret + gamma_full * (roe_t - roe_t.mean()) + noise_t

    df_t = pd.DataFrame({'ROE': roe_t, 'MktCap': mktcap_t, 'Return': ret_t})
    df_t['Quintile'] = pd.qcut(df_t['ROE'], N_GROUPS, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])

    # 等权
    ew_ret = df_t.groupby('Quintile', observed=True)['Return'].mean()
    ew_spreads.append(ew_ret['Q5'] - ew_ret['Q1'])

    # 市值加权
    def weighted_avg(g):
        return np.average(g['Return'], weights=g['MktCap'])
    vw_ret = df_t.groupby('Quintile', observed=True).apply(weighted_avg)
    vw_spreads.append(vw_ret['Q5'] - vw_ret['Q1'])

ew_arr = np.array(ew_spreads)
vw_arr = np.array(vw_spreads)

ew_t = ew_arr.mean() / (ew_arr.std(ddof=1) / np.sqrt(N_MONTHS))
vw_t = vw_arr.mean() / (vw_arr.std(ddof=1) / np.sqrt(N_MONTHS))

print(f"\n  {'Weighting':<12} {'Mean Spread':>14} {'t-stat':>10} {'Significant':>12}")
print("  " + "─" * 52)
print(f"  {'Equal':<12} {ew_arr.mean():>14.4f}% {ew_t:>10.2f} {'✅' if abs(ew_t) > 2 else '❌':>12}")
print(f"  {'Value':<12} {vw_arr.mean():>14.4f}% {vw_t:>10.2f} {'✅' if abs(vw_t) > 2 else '❌':>12}")

print(f"\n💡 解读：")
if abs(vw_t) > abs(ew_t):
    print(f"  市值加权下盈利因子更显著（t = {vw_t:.2f} vs {ew_t:.2f}）")
    print(f"  原因：市值加权放大了高 ROE 大市值股票的权重")
else:
    print(f"  等权下盈利因子更显著（t = {ew_t:.2f} vs {vw_t:.2f}）")

---

## 7. Fama-MacBeth 回归检验

### 📐 Fama-MacBeth 两步回归

**第一步**：每月做截面回归
$$r_{i,t} = \alpha_t + \lambda_t \cdot \text{ROE}_{i,t} + \varepsilon_{i,t}$$

**第二步**：对 $\lambda_t$ 时间序列做 T 检验
$$\bar{\lambda} = \frac{1}{T} \sum_{t=1}^{T} \lambda_t$$

如果 $\bar{\lambda}$ 显著大于 0，说明 ROE 对股票收益有正向预测能力。

In [ ]:
# ========== Fama-MacBeth 回归 ==========
np.random.seed(42)

print("📊 Fama-MacBeth 回归检验盈利因子")
print("─" * 55)

fm_lambdas = []
fm_t_stats = []

for t in range(N_MONTHS):
    base_ret = np.random.normal(1.0, 3.0)
    roe_t = np.random.normal(10, 8, N_STOCKS)
    noise_t = np.random.normal(0, 5, N_STOCKS)
    ret_t = base_ret + gamma_full * (roe_t - roe_t.mean()) + noise_t

    # 截面回归: r_i = alpha + lambda * ROE_i + eps
    X = sm.add_constant(roe_t)
    model = sm.OLS(ret_t, X).fit()
    fm_lambdas.append(model.params[1])  # lambda_t

fm_lambda_arr = np.array(fm_lambdas)
lambda_mean = fm_lambda_arr.mean()
lambda_std = fm_lambda_arr.std(ddof=1)
lambda_se = lambda_std / np.sqrt(N_MONTHS)
fm_t = lambda_mean / lambda_se
fm_p = 2 * (1 - stats.t.cdf(abs(fm_t), df=N_MONTHS - 1))

print(f"\n  📐 FM 回归结果:")
print(f"  λ 均值  = {lambda_mean:.4f}")
print(f"  λ 标准差 = {lambda_std:.4f}")
print(f"  T 统计量 = {fm_t:.4f}")
print(f"  P 值     = {fm_p:.6f}")

# 与分组检验对比
print(f"\n  📊 与分组检验对比:")
print(f"  分组法 RMW t-stat = {t_stat:.4f}")
print(f"  FM 回归 λ t-stat  = {fm_t:.4f}")
print(f"  💡 两种方法结论 {'一致' if (t_stat > 2) == (fm_t > 2) else '不一致'}")

In [ ]:
# ========== 可视化: FM 回归系数时间序列 ==========
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(range(N_MONTHS), fm_lambda_arr, color='steelblue', alpha=0.7, linewidth=1)
ax.axhline(y=lambda_mean, color='red', linestyle='--', linewidth=2, label=f'Mean = {lambda_mean:.4f}')
ax.axhline(y=0, color='black', linewidth=0.5)

# 置信区间
t_crit = stats.t.ppf(0.975, df=N_MONTHS - 1)
ci_lo = lambda_mean - t_crit * lambda_se
ci_hi = lambda_mean + t_crit * lambda_se
ax.fill_between(range(N_MONTHS), ci_lo, ci_hi, alpha=0.15, color='red', label='95% CI')

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('FM Coefficient (\u03bb)', fontsize=12)
ax.set_title('Fama-MacBeth Regression Coefficient for ROE', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

textstr = f'\u03bb mean = {lambda_mean:.4f}\nt-stat = {fm_t:.2f}\np-value = {fm_p:.4f}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=props)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  FM 回归系数 λ 的时间序列")
print(f"  λ 均值 = {lambda_mean:.4f}，{'显著大于 0' if fm_p < 0.05 else '不显著'}")
print(f"  说明 ROE 对股票收益 {'有' if fm_p < 0.05 else '没有'}显著的正向预测能力")

---

## 8. 盈利因子的其他维度

### 📖 书中的讨论

除了 ROE，学术界还研究了盈利的其他维度：

| 维度 | 代表变量 | 核心发现 |
|------|---------|----------|
| **盈利水平** | ROE, ROA, GP | 盈利越高，收益越高 |
| **盈利质量** | 应计利润比率 | 现金流含量越高，盈利质量越好（Sloan 1996）|
| **盈利波动** | 过去 8 个季度 ROE 标准差 | 波动越大，持续性越差（Dichev & Tang 2009）|
| **盈利增长** | SUE, ROA 同比变化 | 增长越快，收益越高（Piotroski 2000）|
| **质量因子** | 综合盈利+成长+安全 | 解释巴菲特长期表现（Frazzini et al. 2018）|

In [ ]:
# ========== 盈利质量: 应计利润 vs 现金流 ==========
np.random.seed(42)

print("📊 盈利质量: 应计异象 (Accruals Anomaly)")
print("─" * 55)
print("\n  Sloan (1996): 盈利 = 现金流 + 应计利润")
print("  盈利的现金流含量越高 → 盈利质量越好 → 收益越高")
print("  应计利润越高 → 盈利质量越差 → 收益越低")

# 模拟应计异象
accrual_spreads = []
for t in range(N_MONTHS):
    base_ret = np.random.normal(1.0, 3.0)
    accrual_t = np.random.normal(0, 5, N_STOCKS)  # 应计利润比率
    noise_t = np.random.normal(0, 5, N_STOCKS)
    # 应计利润效应为负（高应计 → 低收益）
    ret_t = base_ret - 0.08 * (accrual_t - accrual_t.mean()) + noise_t

    df_t = pd.DataFrame({'Accrual': accrual_t, 'Return': ret_t})
    df_t['Group'] = pd.qcut(df_t['Accrual'], 5, labels=['Q1(低)', 'Q2', 'Q3', 'Q4', 'Q5(高)'])
    gret = df_t.groupby('Group', observed=True)['Return'].mean()
    accrual_spreads.append(gret['Q1(低)'] - gret['Q5(高)'])  # 做多低应计，做空高应计

acc_arr = np.array(accrual_spreads)
acc_t = acc_arr.mean() / (acc_arr.std(ddof=1) / np.sqrt(N_MONTHS))
print(f"\n  应计异象 Spread 均值 = {acc_arr.mean():.4f}%")
print(f"  T 统计量 = {acc_t:.2f}")
print(f"  {'✅ 显著' if abs(acc_t) > 2 else '❌ 不显著'}")

---

## 📌 核心概念回顾

### 📌 盈利因子 (RMW)
- **定义**: 做多高盈利股票、做空低盈利股票的多空组合
- **公式**: $\text{RMW} = R_{\text{High ROE}} - R_{\text{Low ROE}}$
- **含义**: 高盈利公司有更高的预期收益率
- **判断标准**: T 统计量 > 2 表示显著

### 📌 盈利指标选择
- **ROE**: 最常用，FF5 采用
- **ROA**: 剔除杠杆影响
- **GP（毛利润/总资产）**: Novy-Marx (2013) 推荐
- **RNOA**: 理论最优，但近年表现不佳

### 📌 等权 vs 市值加权
- **等权**: 低 ROE 组中小市值被提升权重 → Spread 缩小
- **市值加权**: 高 ROE 大市值权重更大 → Spread 扩大
- **本质**: ROE 与市值正相关，低 ROE 公司往往是小市值

### 📌 双重排序
- **目的**: 控制市值对盈利因子的干扰
- **方法**: 先按市值分组，再在组内按 ROE 分组
- **发现**: 控制市值后，盈利因子在大多数市值组内均显著

### 🔗 完整流程
```
数据准备 → ROE 计算 → 截面分组 → 计算 Spread → T 检验 → 单调性检验
    ↓           ↓          ↓          ↓          ↓         ↓
  财务数据    TTM 处理    5/10 组    Q5-Q1    t > 2?   Spearman ρ
```

### 📝 检验指标汇总

| 指标 | 公式 | 含义 | 判断标准 |
|------|------|------|----------|
| T 统计量 | $\bar{x} / SE$ | 均值是否显著不为 0 | \|t\| > 2 |
| Sharpe Ratio | $\bar{x} / s$ | 风险调整后收益 | > 0.5 较好 |
| Spearman ρ | 秩相关系数 | 单调性 | > 0.8 强 |
| FM λ | 截面回归系数 | 因子溢价 | 显著 > 0 |

---

## ❌ 常见误区

### ❌ 误区 1: 高盈利公司的股票一定涨
**✓ 正确理解**: 盈利因子说的是"高盈利公司**平均而言**有更高的预期收益"，但单只股票、单个月份可能完全相反。因子溢价是统计规律，不是确定性结论。

### ❌ 误区 2: ROE 越高越好，直接买最高 ROE 的股票
**✓ 正确理解**: 极高的 ROE 可能来自高杠杆（如金融股）或不可持续的一次性收益。应该用 ROE(TTM) 并注意行业差异。此外，书中指出应控制市值后再看盈利因子的效果。

### ❌ 误区 3: 等权和市值加权的结论应该一样
**✓ 正确理解**: 两种加权方式会给出不同结论。等权下低 ROE 组中的小市值股票权重被放大，导致低 ROE 组收益偏高、Spread 缩小。书中实证表明市值加权下盈利因子更显著。

### ❌ 误区 4: 盈利因子在所有市场都有效
**✓ 正确理解**: 盈利因子在 A 股有效（书中实证），但投资因子在 A 股不显著。不同市场的因子表现可能差异很大，不能简单套用美股结论。

### ❌ 误区 5: 只看 ROE 就够了
**✓ 正确理解**: 盈利有很多维度——水平（ROE）、质量（应计利润）、波动（持续性）、增长（SUE）。书中提到 Novy-Marx (2013) 认为 GP 的预测能力与 BM 相当，且优于 ROE。单一指标可能遗漏信息。